In [107]:
# =========================
#version_6
# N-Hits model
#loss -> Huber(weighted) + smape 
#Early stopping
# =========================

import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F

import matplotlib.pyplot as plt
from korean_lunar_calendar import KoreanLunarCalendar
from numpy.lib.stride_tricks import sliding_window_view

from copy import deepcopy

from collections import defaultdict



plt.rcParams['font.family'] = 'AppleGothic'  # macOS
plt.rcParams['axes.unicode_minus'] = False  # 하이픈으로 대체


########### Util ################
# === 미래 달력 유틸 ===
def build_holiday_index(solar_md_holidays, lunar_solar_dates, years):
    # 양력 고정일(연도별 생성) + 음력 변환일(이미 ISO들)
    solar_list = []
    for y in years:
        for (m, d) in solar_md_holidays:
            try:
                solar_list.append(pd.Timestamp(year=y, month=m, day=d))
            except Exception:
                pass
    lunar_list = pd.to_datetime(lunar_solar_dates, errors='coerce')
    holi = pd.to_datetime(pd.Index(solar_list).append(pd.Index(lunar_list)).unique()).sort_values()
    return holi

def make_future_calendar(last_dates: np.ndarray, horizon: int,
                         holiday_index: pd.DatetimeIndex, K: int = 10):
    """
    last_dates: np.ndarray[pd.Timestamp] shape (S,)
    return: future_wd(S,H), future_mm(S,H), future_px(S,H)
    """
    S = len(last_dates)
    base = last_dates.astype('datetime64[D]')[:, None]                 # (S,1)
    offs = np.arange(1, horizon + 1, dtype='timedelta64[D]')[None, :]  # (1,H)
    fut = (base + offs).astype('datetime64[D]')                         # (S,H)
    fut_pd = pd.to_datetime(fut.reshape(-1))                            # (S*H,)

    # 요일/월(0~11)
    wd = fut_pd.dayofweek.values.reshape(S, horizon).astype(np.int64)   # (S,H)
    mm = (fut_pd.month.values.reshape(S, horizon) - 1).astype(np.int64) # (S,H)

    # 휴일 근접도
    if len(holiday_index) == 0:
        prox = np.zeros((S, horizon), dtype=np.float32)
    else:
        fut_d = fut_pd.values.astype('datetime64[D]')[:, None]
        hol_d = holiday_index.values.astype('datetime64[D]')
        dist = np.abs(fut_d - hol_d).astype('timedelta64[D]').astype(np.int32)
        mind = dist.min(axis=1)
        mind = np.clip(mind, 0, K)
        prox = ((K - mind) / float(K)).astype(np.float32).reshape(S, horizon)
    return wd, mm, prox
def ensure_time_major(x: torch.Tensor, lookback: int, n_features: int) -> torch.Tensor:
    """
    x를 (B, T, F)로 강제.
    - (B, T, F)이면 그대로
    - (B, F, T)이면 permute(0,2,1)
    - 그 외는 에러
    """
    if x.dim() != 3:
        raise ValueError(f"Expected 3D tensor, got {x.dim()}D: {tuple(x.shape)}")
    B, A, B2 = x.shape
    if A == lookback and B2 == n_features:
        return x
    if A == n_features and B2 == lookback:
        return x.permute(0, 2, 1).contiguous()
    raise ValueError(f"Unexpected shape for sequence tensor: {tuple(x.shape)} (T={lookback}, F={n_features})")

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode='min',
                 restore_best_weights=True, min_epochs=0,
                 relative=True, smooth_beta=0.0):
        """
        min_delta: 개선폭 기준 (relative=True면 비율, e.g., 0.005 = 0.5%)
        min_epochs: 이 에폭 전에는 중단 금지
        smooth_beta: 0~1, >0이면 EMA로 스무딩한 값을 모니터 (0이면 원시값)
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.restore_best_weights = restore_best_weights
        self.min_epochs = min_epochs
        self.relative = relative
        self.smooth_beta = smooth_beta

        self.best = None
        self.best_state = None
        self.wait = 0
        self.stop = False
        self.ema = None   # for smoothing

    def _improved(self, current):
        if self.best is None:
            return True
        # 상대/절대 개선폭 계산
        if self.mode == 'min':
            if self.relative:
                need = self.best * (1.0 - self.min_delta)
                return current < need
            else:
                return current < (self.best - self.min_delta)
        else:
            if self.relative:
                need = self.best * (1.0 + self.min_delta)
                return current > need
            else:
                return current > (self.best + self.min_delta)

    def step(self, current, model, epoch_idx: int):
        # 선택: 스무딩
        if self.smooth_beta > 0.0:
            self.ema = current if self.ema is None else (self.smooth_beta*self.ema + (1-self.smooth_beta)*current)
            monitor_val = self.ema
        else:
            monitor_val = current

        if self.best is None or self._improved(monitor_val):
            self.best = monitor_val
            self.best_state = deepcopy(model.state_dict())
            self.wait = 0
            return False

        # 아직 최소 에폭 전이면 기다림만 증가
        self.wait += 1
        if epoch_idx+1 < self.min_epochs:
            return False

        if self.wait >= self.patience:
            self.stop = True
            if self.restore_best_weights and self.best_state is not None:
                model.load_state_dict(self.best_state)
            return True
        return False
# === 공통 피처 정의 (학습/추론 동일) ===
FEATURES = [
    'clipped_SQ','rolling_mean_7','delta_scaled',
    # 월/연 Fourier (k=1..3)
    'month_sin1','month_cos1','month_sin2','month_cos2','month_sin3','month_cos3',
    'doy_sin1','doy_cos1','doy_sin2','doy_cos2','doy_sin3','doy_cos3',
    'holiday_prox','is_holiday',
    'w_sin1','w_sin2','w_cos1','w_cos2',
    'lag_7','lag_14','lag_28',
    'rel_level_7','rel_level_14',
    'vol_7','vol_14',
    'momentum_7','momentum_14',
    'ewm_mean_7','ewm_mean_14',
    'holiday_prox_lag1', 'holiday_prox_lag2', 'holiday_prox_lag3',
    'holiday_prox_lead1','holiday_prox_lead2','holiday_prox_lead3',
    'holiday_prox_lead4','holiday_prox_lead5','holiday_prox_lead6','holiday_prox_lead7'
]
FEAT = {name: i for i, name in enumerate(FEATURES)}
for k in ['rolling_mean_7','holiday_prox','momentum_7','vol_14','ewm_mean_7']:
    assert k in FEAT, f"Missing feature index: {k}"
    
def build_features(
    df: pd.DataFrame,
    scaler_y: MinMaxScaler,
    scaler_rm : MinMaxScaler,
    scaler_delta: MinMaxScaler,
    fit: bool = False,
    date_col: str = '영업일자'
) -> pd.DataFrame:
    """
    학습/추론에서 동일하게 쓰는 피처 빌더.
    - clipped_SQ, rolling_mean_7 scaling
    - delta_scaled scaling
    - Fourier/holiday proximity/weekly sincos
    - ts-stats (lag/roll/ewm/momentum/rel_level/vol) + 결측 처리
    """
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # 요일/월/시즌
    out['weekday'] = out[date_col].dt.dayofweek.astype(int)
    m = out[date_col].dt.month.astype(np.int16)
    out['season'] = m.map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}).astype(int)

    # 휴일, Fourier, 주간 주기
    out = generate_combined_holiday_list(out, solar_md_holidays, lunar_solar_dates)
    out = add_fourier_seasonal_features(out, date_col=date_col)

    t = (out[date_col] - out[date_col].min()).dt.days.values
    for k in (1, 2):
        out[f'w_sin{k}'] = np.sin(2*np.pi*k*t/7).astype('float32')
        out[f'w_cos{k}'] = np.cos(2*np.pi*k*t/7).astype('float32')

    # holiday proximity (+ lag/lead)
    out = add_holiday_proximity(out, date_col, 'is_holiday', 'holiday_prox', K=10, return_what='prox')
    # 먼저 기존 lag/lead 컬럼이 있으면 정리(덮어쓰기 혼선 방지)
    _drop_cols = [f'holiday_prox_lag{k}' for k in range(1, PREDICT+1)] + \
                [f'holiday_prox_lead{k}' for k in range(1, PREDICT+1)]
    exist_drop = [c for c in _drop_cols if c in out.columns]
    if exist_drop:
        out.drop(columns=exist_drop, inplace=True)

    # 벡터화로 한 번에 생성 (1..PREDICT 모두)
    lag_cols  = {f'holiday_prox_lag{k}':  out['holiday_prox'].shift(k)   for k in range(1, PREDICT+1)}
    lead_cols = {f'holiday_prox_lead{k}': out['holiday_prox'].shift(-k)  for k in range(1, PREDICT+1)}
    out = out.assign(**lag_cols, **lead_cols)

    # 결측/타입 정리
    mk_cols = [f'holiday_prox_lag{k}' for k in range(1, PREDICT+1)] + \
            [f'holiday_prox_lead{k}' for k in range(1, PREDICT+1)]
    out[mk_cols] = out[mk_cols].fillna(0.0).astype('float32')
    
    # clip / delta / rolling
    out['clipped_SQ']     = clip_df(out['매출수량']) if '매출수량' in out.columns else out['clipped_SQ']
    out['delta']          = out['clipped_SQ'].diff().fillna(0)
    out['rolling_mean_7'] = out['clipped_SQ'].rolling(window=7, min_periods=1).mean()

    # scaling
    if fit:
        scaler_y.fit(out[['clipped_SQ']])
        scaler_rm.fit(out[['rolling_mean_7']])
        scaler_delta.fit(out[['delta']])
    out[['clipped_SQ']] = scaler_y.transform(out[['clipped_SQ']])
    out[['rolling_mean_7']] = scaler_rm.transform(out[['rolling_mean_7']])
    out[['delta_scaled']] = scaler_delta.transform(out[['delta']])

    # ts-stats (누수 안전 옵션: 현재값 제외하려면 shift(1) 사용)
    # 엄격 모드 예시:
    # out_ts = add_ts_stats(out.copy(), target_col="clipped_SQ", date_col=date_col, ...)
    # 부분만 교체하려면 add_ts_stats 내부에서 rolling을 x.shift(1).rolling(...)로 바꾸세요.
    out = add_ts_stats(out, target_col="clipped_SQ", date_col=date_col,
                       lags=(1,7,14,28), roll_windows=(7,14,28), ewm_spans=(7,14))

    # ✅ 모든 시계열 통계 파생 컬럼에서 NaN/Inf 제거
    ts_cols = [c for c in out.columns if c.startswith((
        'lag_', 'momentum_', 'roll_mean_', 'roll_std_', 'rel_level_', 'vol_', 'ewm_mean_'
    ))]
    out[ts_cols] = (out[ts_cols]
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(0.0)
                    .astype('float32'))

    return out
def add_ts_stats(
    df: pd.DataFrame,
    target_col: str = "clipped_SQ",   # 스케일 전(or 스케일 후) 타깃 중 택1
    date_col: str = "영업일자",
    lags = (1, 7, 14, 28),
    roll_windows = (7, 14, 28),
    ewm_spans = (7, 14),
    out_prefix: str = "",
    eps: float = 1e-3
) -> pd.DataFrame:
    """
    시계열 통계 피처 생성 (모두 과거만 사용)
    생성: lag_k, roll_mean_k, roll_std_k, ewm_mean_s, momentum_k, rel_level_k, vol_k
    - momentum_k  = (x_t - x_{t-k}) / (|x_{t-k}|+eps)
    - rel_level_k = x_t / (roll_mean_k + eps)
    - vol_k       = roll_std_k / (roll_mean_k + eps)
    """
    # 정렬 보장
    df = df.sort_values(date_col)
    x = df[target_col].astype("float32")

    # Lags
    for k in lags:
        df[f"{out_prefix}lag_{k}"] = x.shift(k).astype("float32")

    # Rolling mean/std (과거 window, 현재 포함 → 누수 방지 위해 shift(1) 후 rolling도 가능)
    for w in roll_windows:
        # 현재 시점 포함 롤링 → 일반적으로 OK. 더 엄격히 하려면 아래 두 줄을 교체:
        #    base = x.shift(1) ; df[f"roll_mean_{w}"] = base.rolling(w, min_periods=1).mean()
        df[f"{out_prefix}roll_mean_{w}"] = x.rolling(w, min_periods=1).mean().astype("float32")
        df[f"{out_prefix}roll_std_{w}"]  = x.rolling(w, min_periods=1).std().fillna(0).astype("float32")

    # EWMA
    for s in ewm_spans:
        df[f"{out_prefix}ewm_mean_{s}"] = x.ewm(span=s, adjust=False).mean().astype("float32")

    # Momentum & Relative level & Volatility (대표 window=7 사용; 필요시 반복문 확장)
    for k in lags:
        df[f"{out_prefix}momentum_{k}"] = ((x - x.shift(k)) / (np.abs(x.shift(k)) + eps)).astype("float32")
    for w in roll_windows:
        m = df[f"{out_prefix}roll_mean_{w}"]
        s = df[f"{out_prefix}roll_std_{w}"]
        df[f"{out_prefix}rel_level_{w}"] = (x / (m + eps)).astype("float32")      # 수준/평균
        df[f"{out_prefix}vol_{w}"]       = (s / (m + eps)).astype("float32")      # 변동성/평균(무단위)

    return df
#################################

#Fixed Random Seed  & Setting Hyperparameter
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seed(42)

LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 32, 50
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass
MONTH_SCALE = 12

MIN_SEQUENCE_COUNT = 10

USE_SMAPE = True
PATIENCE = 10
MIN_DELTA = 0.0
MAX_LAG = 28

# 키: '영업장명_메뉴명', 값: 'YYYY-MM-DD' (ISO 문자열 또는 datetime)
DISCONTINUED = {
    '담하_꼬막_비빔밥': '2024-04-01',
    '담하_들깨_양지탕': '2024-04-01',
    # ...
}

def get_lunar_to_solar(years, lunar_month, lunar_day, span=1):
    calendar = KoreanLunarCalendar()
    dates = []
    for year in years:
        for offset in range(-span, span+1):
            try:
                calendar.setLunar(year, lunar_month, lunar_day + offset, False)
                dates.append(calendar.SolarIsoFormat())
            except:
                pass  # 예외 처리: 음력 마지막날 초과
    return dates
# 예시: 2023 ~ 2025
years = [2023, 2024, 2025]
lunar_solar_dates = []
lunar_solar_dates += get_lunar_to_solar(years, 1, 1, span=1)   # 설날 ±1
lunar_solar_dates += get_lunar_to_solar(years, 8, 15, span=1)  # 추석 ±1

solar_md_holidays = [
    (1, 1),   # 신정
    (3, 1),   # 삼일절
    (5, 5),   # 어린이날
    (6, 6),   # 현충일
    (8, 15),  # 광복절
    (10, 3),  # 개천절
    (10, 9),  # 한글날
    (12, 25), # 크리스마스
]

def generate_combined_holiday_list(df, solar_md_list, lunar_solar_list):
    df = df.copy()
    df['영업일자'] = pd.to_datetime(df['영업일자'])

    # 양력 기반 holiday 판별
    df['is_solar_holiday'] = df['영업일자'].apply(
        lambda x: (x.month, x.day) in solar_md_list
    )

    # 음력 변환된 holiday 포함
    lunar_set = set(pd.to_datetime(lunar_solar_list))
    df['is_lunar_holiday'] = df['영업일자'].isin(lunar_set)

    # 최종 통합
    df['is_holiday'] = (df['is_solar_holiday'] | df['is_lunar_holiday']).astype(int)
    df = df.drop(columns=['is_solar_holiday', 'is_lunar_holiday'])
    return df

def remove_leading_zeros_before_sales(df, min_zero_days=90):
    """
    매출 시작 전 연속 0이 일정 기간 이상이면, 그 전 구간 제거
    (단일 메뉴-업장 그룹 DataFrame을 가정)
    """
    sales_started = df['매출수량'] > 0
    if not sales_started.any():
        return df  # 매출이 전혀 없는 경우 그대로 반환

    first_sale_idx = sales_started.idxmax()

    # 매출 시작 전 구간이 충분히 긴 0으로 구성되어 있다면 제거
    df_before = df.loc[:first_sale_idx - 1]
    if len(df_before) >= min_zero_days and (df_before['매출수량'] == 0).all():
        return df.loc[first_sale_idx:]  # 매출 시작부터 반환
    else:
        return df  # 그대로 반환


def _extract_store_name(g: pd.DataFrame) -> str:
    """
    그룹 g에서 업장명 추출:
    - '영업장명' 컬럼이 있으면 그 값을 사용
    - 없으면 '영업장명_메뉴명'에서 첫 '_' 앞을 업장명으로 간주
    """
    if '영업장명' in g.columns:
        return str(g['영업장명'].iloc[0])
    # '영업장명_메뉴명'이 "업장명_메뉴명" 형태라고 가정
    full = str(g['영업장명_메뉴명'].iloc[0])
    return full.split('_', 1)[0]  # '_'가 여러 개여도 첫 구분만 사용


def filter_all_menus_by_leading_zeros(
    train_df: pd.DataFrame,
    min_zero_days: int = 90,
    apply_to_stores: list[str] | None = None,
    exclude_stores: list[str] | None = None,
    group_col: str = '영업장명_메뉴명',
) -> pd.DataFrame:
    """
    모든 메뉴-업장 그룹에 대해 remove_leading_zeros_before_sales를 적용하되,
    특정 업장에만(또는 특정 업장은 제외하고) 적용할 수 있도록 확장.

    Parameters
    ----------
    train_df : 전체 데이터프레임
    min_zero_days : 매출 시작 전 연속 0 최소 일수
    apply_to_stores : 적용 대상 업장명 리스트 (None이면 전 업장 대상)
    exclude_stores : 적용 제외 업장명 리스트 (None이면 제외 없음)
    group_col : 그룹화 기준 컬럼명 (기본: '영업장명_메뉴명')
    """
    parts = []
    apply_set   = set(apply_to_stores) if apply_to_stores is not None else None
    exclude_set = set(exclude_stores)  if exclude_stores  is not None else set()

    # 기존 순서 보존 원하면 sort=False 유지
    for _, g in train_df.groupby(group_col, sort=False):
        store = _extract_store_name(g)

        # 적용 여부 결정
        apply_flag = True
        if apply_set is not None:
            apply_flag = (store in apply_set)
        if store in exclude_set:
            apply_flag = False

        if apply_flag:
            parts.append(remove_leading_zeros_before_sales(g, min_zero_days))
        else:
            parts.append(g)

    if parts:
        return pd.concat(parts, ignore_index=True)
    return train_df.reset_index(drop=True)

# === 추가: Fourier 계절 피처 함수 ===
def add_fourier_seasonal_features(df: pd.DataFrame, date_col: str = '영업일자') -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    m = df[date_col].dt.month.astype(np.int16)          # 1..12
    doy = df[date_col].dt.dayofyear.astype(np.int16)    # 1..365 (윤년은 무시해도 충분)
    # (A) 월 주기: 12개월 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        df[f'month_sin{k}'] = np.sin(2*np.pi*k*m/12).astype('float32')
        df[f'month_cos{k}'] = np.cos(2*np.pi*k*m/12).astype('float32')
    # (B) 연간 주기: 365일 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        df[f'doy_sin{k}'] = np.sin(2*np.pi*k*doy/365).astype('float32')
        df[f'doy_cos{k}'] = np.cos(2*np.pi*k*doy/365).astype('float32')
    # (C) 범주형 월 인덱스(임베딩용)
    df['month_idx'] = (m - 1).astype(int)  # 0~11
    return df

#################Loss######################### 
import numpy as np

def estimate_delta_from_y(y_real, clip_min: float = 10.0, clip_max: float = 60.0) -> float:
    """
    y_real: 실측 스케일(예: 매출수량) 1D 배열
    반환값: Huber loss의 delta (float)

    - 중앙값 기준 MAD(중앙절대편차)로 노이즈 스케일을 추정
    - 권장 상수 1.35 * MAD 사용
    - 지나치게 작거나 큰 delta는 clip_min ~ clip_max로 클리핑
    """
    y = np.asarray(y_real, dtype=float)
    y = y[np.isfinite(y)]
    if y.size == 0:
        return float((clip_min + clip_max) / 2.0)

    med = np.median(y)
    mad = np.median(np.abs(y - med))

    # 너무 작은 MAD 보호
    mad = max(mad, 1e-6)

    delta = 1.35 * mad
    delta = float(np.clip(delta, clip_min, clip_max))
    return delta


def _slugify(text: str) -> str:
    # 파일명 안전 문자열
    text = re.sub(r'[^\w\-_. ]', '_', text)
    return re.sub(r'\s+', '_', text).strip('_')[:80]

def weighted_huber_smape(
    yhat, y, w=None, *, delta=0.05, eps=1e-3, alpha=0.5
):
    hub = F.huber_loss(yhat, y, delta=delta, reduction='none')
    den = yhat.abs() + y.abs()
    if isinstance(eps, float) or isinstance(eps, int):
        den = torch.clamp(den, min=float(eps))
    else:
        den = torch.maximum(den, eps)
    smp = 2.0 * (yhat - y).abs() / den
    loss = alpha * hub + (1 - alpha) * smp
    if w is not None: loss = loss * w
    return loss.mean()

def smape_loss(pred, target, eps=1e-3, reduction='mean', ignore_zero_target=False):
    """
    ignore_zero_target=True이면 target==0인 위치는 sMAPE 평균에서 제외.
    (대회 채점 규칙과 정렬)
    """
    num = (pred - target).abs()
    den = (pred.abs() + target.abs()).clamp_min(eps)
    sm = 2.0 * num / den  # (B, PREDICT)

    if reduction == 'none' and not ignore_zero_target:
        return sm

    if ignore_zero_target:
        mask = (target != 0).float()                     # (B,P)
        # 각 샘플(행)별 유효 포인트 평균 → 배치 평균
        denom = mask.sum(dim=1).clamp_min(1.0)           # (B,)
        row_mean = (sm * mask).sum(dim=1) / denom        # (B,)
        return row_mean.mean() if reduction != 'none' else (sm, mask)

    # 기존 동작
    return sm.mean()


def save_mse_curves(history, title="MSE Curve", out_dir="./loss_plots_mse", filename="mse_curve.png"):
    train_mse = list(history.get('train_mse', []))
    val_mse   = list(history.get('val_mse', []))
    L = max(len(train_mse), len(val_mse))
    train_mse += [None]*(L - len(train_mse))
    val_mse   += [None]*(L - len(val_mse))

    plt.figure(figsize=(8,5), dpi=140)
    plt.plot(range(1,L+1), train_mse, marker='o', label='Train MSE')
    plt.plot(range(1,L+1), val_mse,   marker='o', label='Validation MSE')
    plt.title(title); plt.xlabel('Epoch'); plt.ylabel('MSE (real scale)')
    plt.grid(alpha=0.4, linestyle='--'); plt.legend()
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, filename)
    plt.savefig(path, bbox_inches='tight'); plt.close()
    return path
####################################################

def inverse_clipped_from_scaler(scaler_xy: MinMaxScaler, scaled_vals: np.ndarray) -> np.ndarray:
    """
    scaler_xy는 ['clipped_SQ','rolling_mean_7']에 대해 fit 되어 있음.
    clipped_SQ만 역변환하려면 2열 dummy를 만들어 1열만 복원.
    """
    dummy = np.zeros((len(scaled_vals), 2), dtype=np.float32)
    dummy[:, 0] = scaled_vals
    inv = scaler_xy.inverse_transform(dummy)[:, 0]
    return inv
def add_holiday_proximity(
    df: pd.DataFrame,
    date_col: str = '영업일자',
    holiday_col: str = 'is_holiday',
    out_col: str = 'holiday_prox',
    K: int = 7,
    return_what: str = 'prox',  # 'prox' 또는 'dist'
) -> pd.DataFrame:
    """
    캘린더 휴일 기준으로 각 날짜가 휴일에 얼마나 근접했는지 계산합니다.
    - prox: (K - min(dist_prev, dist_next)) / K ∈ [0,1], 당일 휴일=1, K일 이상 떨어지면 0
    - dist: min(dist_prev, dist_next) ∈ [0, K] (K로 클리핑)

    Notes
    -----
    * holiday_col은 미래를 '알 수 있는' 캘린더 정보이므로 누수 위험이 없습니다.
    * df의 원래 행 순서를 유지합니다.
    """
    if date_col not in df.columns:
        raise KeyError(f"'{date_col}' not in df")
    if holiday_col not in df.columns:
        raise KeyError(f"'{holiday_col}' not in df")

    # 원래 인덱스 저장
    orig_index = df.index

    # 날짜 정렬본으로 계산
    tmp = df[[date_col, holiday_col]].copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    tmp = tmp.sort_values(date_col)

    mask = tmp[holiday_col].astype(bool)
    # 휴일이면 그 날짜, 아니면 NaT
    s_h = tmp[date_col].where(mask)

    # 과거/미래 휴일 날짜
    prev_h = s_h.ffill()
    next_h = s_h.bfill()

    # 거리 계산(일수)
    dist_prev = (tmp[date_col] - prev_h).dt.days.astype('float32')
    dist_next = (next_h - tmp[date_col]).dt.days.astype('float32')

    # 휴일이 아예 없을 때 NaN → K+1로 대체
    dist_prev = dist_prev.fillna(K + 1)
    dist_next = dist_next.fillna(K + 1)

    # 최소 거리 후 K로 클리핑
    dist_h = np.minimum(dist_prev, dist_next).clip(0, K).astype('float32')

    if return_what == 'dist':
        out = dist_h
    elif return_what == 'prox':
        # 근접도: 0(멀다) ~ 1(당일 휴일)
        out = ((K - dist_h) / K).astype('float32')
    else:
        raise ValueError("return_what must be 'prox' or 'dist'")

    # 정렬 전 순서로 복원
    out = out.reindex(tmp.index)                # 안전: 이미 tmp와 동일
    out_df = pd.DataFrame({out_col: out}, index=tmp.index)
    out_df = out_df.reindex(orig_index)         # 원래 df 순서로

    # 원본 df에 컬럼으로 추가
    df[out_col] = out_df[out_col].values.astype('float32')
    return df

class MRBlock(nn.Module):
    def __init__(self, in_dim, horizon, pool_k: int, mlp_dim=128, mlp_layers=2):
        super().__init__()
        self.pool_k = pool_k
        layers = [nn.Linear(in_dim, mlp_dim), nn.ReLU()]
        for _ in range(mlp_layers-1):
            layers += [nn.Linear(mlp_dim, mlp_dim), nn.ReLU()]
        self.mlp = nn.Sequential(*layers)
        # ↓ proj를 두 배 입력(평균+최대)로 바꿈
        self.proj = nn.Linear(mlp_dim * 2, horizon)
        self.norm = nn.LayerNorm(mlp_dim)
        self.drop = nn.Dropout(p=0.1)

    def forward(self, x):                # x: (B,T,F)
        x_ch = x.transpose(1, 2)         # (B,F,T)
        if self.pool_k > 1:
            x_pool = F.avg_pool1d(x_ch, kernel_size=self.pool_k, stride=self.pool_k, ceil_mode=True)
        else:
            x_pool = x_ch
        x_pool = x_pool.transpose(1, 2)  # (B,T',F)
        z = self.mlp(x_pool)             # (B,T',D)
        z = self.norm(z)
        z = self.drop(z)
        # 평균 + 최대 (시간축)
        z_mean = z.mean(dim=1)           # (B,D)
        z_max  = z.amax(dim=1)           # (B,D)
        z_cat  = torch.cat([z_mean, z_max], dim=-1)  # (B,2D)
        h = self.proj(z_cat)             # (B,H)
        return h

class NHiTSWithEmbeddingMR(nn.Module):
    def __init__(self, lookback, input_dim, horizon,
                 pools=(28,7,3,1),
                 weekday_vocab=7, weekday_emb_dim=2,
                 season_vocab=4,  season_emb_dim=2,
                 month_vocab=12,  month_emb_dim=3,
                 emb_dropout=0.25, use_sigmoid_output=False):
        super().__init__()
        self.lookback, self.horizon = lookback, horizon
        self.use_sigmoid_output = use_sigmoid_output
        self.use_residual_base = False      # 필요할 때 외부에서 True로
        self.r_scale = 0.25                 # 비율 경로 스케일(튜닝 가능)

        # ---- 임베딩들 ----
        self.weekday_emb = nn.Embedding(weekday_vocab, weekday_emb_dim)
        self.season_emb  = nn.Embedding(season_vocab,  season_emb_dim)
        self.month_emb   = nn.Embedding(month_vocab,   month_emb_dim)

        self._in_cat = weekday_emb_dim + season_emb_dim + month_emb_dim
        self.post_emb_norm = nn.LayerNorm(input_dim + self._in_cat)
        self.post_emb_drop = nn.Dropout(emb_dropout)

        # ---- 멀티해상도 블록 ----
        self.blocks = nn.ModuleList([
            MRBlock(in_dim=input_dim + self._in_cat,
                    horizon=horizon, pool_k=p, mlp_dim=128, mlp_layers=2)
            for p in pools
        ])

        # ---- 미래 보조 임베딩 ----
        self.h_emb = nn.Embedding(horizon, 8)
        self.wd_future_emb = nn.Embedding(7, 3)
        self.mm_future_emb = nn.Embedding(12, 2)
        self.prox_lin = nn.Linear(1, 2)   # 연속 prox → 2차원

        # ---- 고정 in_dim 헤드들 (rep_exp:1 + aux:15 = 16) ----
        aux_dim = 8 + 3 + 2 + 2  # = 15
        in_dim = 1 + aux_dim     # = 16

        self.head = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )
        self.ratio_head = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)               # tanh 전용 스칼라
        )

    def forward(self, x_num, x_weekday, x_season, x_month,
                future_weekday=None, future_month=None, future_prox=None,
                base_last=None):
        # ---- 카테고리 임베딩 결합 ----
        w = self.weekday_emb(x_weekday)
        s = self.season_emb(x_season)
        m = self.month_emb(x_month)
        x = torch.cat([x_num, w, s, m], dim=-1)
        x = self.post_emb_norm(x)
        x = self.post_emb_drop(x)

        # ---- 블록 → 표현(rep) ----
        reps = []
        for b in self.blocks:
            reps.append(b(x))                  # (B,H)
        rep = torch.stack(reps, dim=0).mean(dim=0)  # (B,H) 공용 표현

        B, H = rep.size()
        device = rep.device

        # ---- 미래 캘린더 보조 ----
        if future_weekday is None:
            future_weekday = (x_weekday[:, -1].unsqueeze(1) + torch.arange(1, H+1, device=device)) % 7
        if future_month is None:
            future_month = x_month[:, -1].unsqueeze(1).expand(B, H)
        if future_prox is None:
            future_prox = torch.zeros(B, H, device=device)

        he = self.h_emb(torch.arange(H, device=device).unsqueeze(0).expand(B, -1))  # (B,H,8)
        we = self.wd_future_emb(future_weekday)                                     # (B,H,3)
        me = self.mm_future_emb(future_month)                                       # (B,H,2)
        pe = self.prox_lin(future_prox.unsqueeze(-1))                               # (B,H,2)

        aux = torch.cat([he, we, me, pe], dim=-1)   # (B,H,15)
        rep_exp = rep.unsqueeze(-1)                 # (B,H,1)
        feat = torch.cat([rep_exp, aux], dim=-1)    # (B,H,16)

        # ---- 메인 예측 ----
        main = self.head(feat).squeeze(-1)          # (B,H)
        if self.use_sigmoid_output:
            main = torch.sigmoid(main)

        # ---- 잔차/비율 경로 (옵션) ----
        if self.use_residual_base:
            if base_last is None:
                base = x_num[:, -1, FEAT['rolling_mean_7']]        # (B,)
            else:
                base = base_last                                   # (B,) 또는 (B,H)
            if base.dim() == 1:
                base = base.unsqueeze(1).expand(B, H)              # (B,H)

            ratio = torch.tanh(self.ratio_head(feat).squeeze(-1))  # (B,H), -1..1
            out = main + base * (self.r_scale * ratio)
        else:
            out = main

        return out

def clip_df(series, q=0.995):
    s = pd.Series(series, copy=False).astype('float32')
    # 결측 허용 분위수 사용
    upper_bound = float(np.nanquantile(s, q))
    # 하한은 건드리지 않고 상한만 잘라냄
    return s.clip(upper=upper_bound)

# =========================
# 1) Train: N-HiTS + Embedding 
# =========================
def train_nhits_embed(train_df, use_validation=True,  
                      lr: float = 1e-3, weight_decay: float = 1e-5,
                      max_grad_norm: float = 1.0,
                      plot_dir: str = './loss_plots',
                      use_sigmoid_output: bool = False,   # 스파이크 추종 위해 기본 False 권장
                      ):
    trained_models = {}

    for store_menu, group in tqdm(train_df.groupby(['영업장명_메뉴명']), desc='Training N-HiTS + Emb'):
        # -------- preproc --------
        key = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))
        store_train = group.sort_values('영업일자').copy()
        store_train['영업일자'] = pd.to_datetime(store_train['영업일자'])

        # 짧은 시계열 제외
        if len(store_train) < LOOKBACK + PREDICT + MIN_SEQUENCE_COUNT:
            continue

        # -------- split 기준 --------
        N = len(store_train)
        if use_validation:
            cutoff_row = max(LOOKBACK, int(round(N * 0.8)))
            cutoff_row = min(cutoff_row, N-1)
        else:
            cutoff_row = N

        # -------- 스케일링 --------
        scaler_y     = MinMaxScaler()
        scaler_rm    = MinMaxScaler()
        scaler_delta = MinMaxScaler()
        train_slice  = slice(0, cutoff_row)

        if cutoff_row < 1:
            continue


        # 1) train-part로 먼저 fit
        train_part = store_train.iloc[:cutoff_row].copy()
        _ = build_features(train_part, scaler_y, scaler_rm,scaler_delta, fit=True, date_col='영업일자')
        # 2) 같은 스케일러로 전체 transform (누수 없음)
        ft = build_features(store_train, scaler_y, scaler_rm, scaler_delta, fit=False, date_col='영업일자')

        # 수치 피처(임베딩 제외)
        ft[FEATURES] = ft[FEATURES].fillna(0.0).astype('float32')


       # 벡터화된 시퀀스 생성
        vals = ft[FEATURES].values.astype(np.float32)       # (N,F)
        tgt  = ft['clipped_SQ'].values.astype(np.float32)   # (N,)
        wd   = ft['weekday'].values.astype(np.int64)
        ss   = ft['season'].values.astype(np.int64)
        mm   = ft['month_idx'].values.astype(np.int64)

        total_seq = len(ft) - LOOKBACK - PREDICT + 1
        if total_seq <= 0: continue

        X_np = sliding_window_view(vals, LOOKBACK, axis=0)[:total_seq]                    # (S,T,F)
        y_np = sliding_window_view(tgt, LOOKBACK + PREDICT, axis=0)[:total_seq, LOOKBACK:]# (S,H)
        wd_np= sliding_window_view(wd, LOOKBACK, axis=0)[:total_seq]                      # (S,T)
        ss_np= sliding_window_view(ss, LOOKBACK, axis=0)[:total_seq]
        mm_np= sliding_window_view(mm, LOOKBACK, axis=0)[:total_seq]

        # === 👉 여기부터 추가: 시퀀스별 last_date + 미래 달력 텐서 사전계산 ===
        dates_np = ft['영업일자'].values.astype('datetime64[ns]')          # (N,)
        win_dates = sliding_window_view(dates_np, LOOKBACK + PREDICT)[:total_seq]  # (S,T+H)
        last_dates = pd.to_datetime(win_dates[:, LOOKBACK - 1])            # (S,) 각 시퀀스 마지막 관측일

        yrs = set(pd.to_datetime(ft['영업일자']).dt.year.tolist())
        yrs.update({min(yrs)-1, max(yrs)+1})                               # 안전 여유연도
        holiday_index = build_holiday_index(solar_md_holidays, lunar_solar_dates, years=sorted(list(yrs)))

        f_wd_np, f_mm_np, f_px_np = make_future_calendar(last_dates.values, PREDICT, holiday_index, K=10)
        f_wd_all = torch.from_numpy(f_wd_np).long().to(DEVICE)    # (S,H)
        f_mm_all = torch.from_numpy(f_mm_np).long().to(DEVICE)    # (S,H)
        f_px_all = torch.from_numpy(f_px_np).float().to(DEVICE)   # (S,H)

        X_num = torch.from_numpy(X_np).float().to(DEVICE)
        X_num = ensure_time_major(X_num, LOOKBACK, len(FEATURES))
        y     = torch.from_numpy(y_np).float().to(DEVICE)
        wd    = torch.from_numpy(wd_np).long().to(DEVICE)
        ss    = torch.from_numpy(ss_np).long().to(DEVICE)
        mm    = torch.from_numpy(mm_np).long().to(DEVICE)
        # ---- 시퀀스 기준 split ----
        if use_validation:
            split_idx = int(len(X_num) * 0.8)
            Xtr, Xval = X_num[:split_idx], X_num[split_idx:]
            ytr, yval = y[:split_idx], y[split_idx:]
            wdtr, wdval = wd[:split_idx], wd[split_idx:]
            sstr,  ssval  = ss[:split_idx],  ss[split_idx:]
            mmtr,   mmval   = mm[:split_idx],   mm[split_idx:]
            # === 👉 추가: 미래 달력 텐서도 동일 분리
            f_wd_tr, f_wd_val = f_wd_all[:split_idx], f_wd_all[split_idx:]
            f_mm_tr, f_mm_val = f_mm_all[:split_idx], f_mm_all[split_idx:]
            f_px_tr, f_px_val = f_px_all[:split_idx], f_px_all[split_idx:]
        else:
            Xtr, ytr = X_num, y
            wdtr, sstr, mmtr = wd, ss, mm
            # === 👉 추가: 전부 학습용
            f_wd_tr, f_mm_tr, f_px_tr = f_wd_all, f_mm_all, f_px_all
            Xval = yval = wdval = ssval = None

        Xtr, ytr = Xtr.to(DEVICE), ytr.to(DEVICE)
        wdtr, sstr , mmtr = wdtr.to(DEVICE), sstr.to(DEVICE) , mmtr.to(DEVICE)
        if use_validation and Xval is not None:
            Xval, yval = Xval.to(DEVICE), yval.to(DEVICE)
            wdval, ssval, mmval = wdval.to(DEVICE), ssval.to(DEVICE), mmval.to(DEVICE)   

        # # ---------- split tensors ----------
        # use_val = use_validation  # 메뉴별 로컬 복사
        # if use_val:
        #     seq_targets = np.asarray(seq_targets)
        #     train_mask  = seq_targets < cutoff_row
        #     val_mask    = ~train_mask
        #     if train_mask.sum() == 0 or val_mask.sum() == 0:
        #         Xtr, ytr, wdtr, sstr, mmtr = X_num, y, wd, ss, mm
        #         use_val = False
        #     else:
        #         Xtr, Xval = X_num[train_mask], X_num[val_mask]
        #         ytr, yval = y[train_mask],     y[val_mask]
        #         wdtr, wdval = wd[train_mask],  wd[val_mask]
        #         sstr, ssval = ss[train_mask],  ss[val_mask]
        #         mmtr, mmval = mm[train_mask],  mm[val_mask]
        # else:
        #     Xtr, ytr, wdtr, sstr, mmtr = X_num, y, wd, ss, mm

        # -------- device ----------
        Xtr, ytr = Xtr.to(DEVICE), ytr.to(DEVICE)
        wdtr, mmtr, sstr = wdtr.to(DEVICE), mmtr.to(DEVICE), sstr.to(DEVICE)
        if use_validation:
            Xval, yval = Xval.to(DEVICE), yval.to(DEVICE)
            wdval, mmval, ssval = wdval.to(DEVICE), mmval.to(DEVICE), ssval.to(DEVICE)

        model = NHiTSWithEmbeddingMR(
            lookback=LOOKBACK, input_dim=len(FEATURES), horizon=PREDICT,
            pools=(28,7,3,1),
            weekday_vocab=7, weekday_emb_dim=2,
            season_vocab=4,  season_emb_dim=2,
            month_vocab=12,  month_emb_dim=3,   # <- 추가
            use_sigmoid_output=use_sigmoid_output,
            emb_dropout=0.25
        ).to(DEVICE)
        model.use_residual_base = True  # 켜기
        
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

        # ---------- 로깅 ----------
        train_losses, val_losses = [], []
        val_smape_list = []
        early = EarlyStopping(
            patience=PATIENCE, min_delta=0.005, mode='min',   # 0.5% 상대개선
            restore_best_weights=True, min_epochs=8, relative=True,
            smooth_beta=0.0  # 노이즈 크면 0.6~0.9로
        )
    

        # 스토어별 y_min/y_max 텐서 (원스케일 SMAPE 계산용)
        # scaler_y.data_min_/data_max_는 shape (1,) 또는 (d,)
        y_min_val = float(scaler_y.data_min_[0]); y_max_val = float(scaler_y.data_max_[0])

        y_train_real = group.sort_values('영업일자')['매출수량'].iloc[:cutoff_row].values
        delta_y = estimate_delta_from_y(y_train_real, clip_min=10.0, clip_max=80.0)

        for ep in range(EPOCHS):
            model.train()
            idx = torch.randperm(len(Xtr))
            sum = 0.0; n_obs = 0
            NOISE_BLOCK = {
                # 결정론/이진
                'is_holiday','holiday_prox',
                'holiday_prox_lag1','holiday_prox_lag2','holiday_prox_lag3',
                'holiday_prox_lead1','holiday_prox_lead2','holiday_prox_lead3',
                # 주기(결정론)
                'w_sin1','w_sin2','w_cos1','w_cos2',
                'month_sin1','month_cos1','month_sin2','month_cos2','month_sin3','month_cos3',
                'doy_sin1','doy_cos1','doy_sin2','doy_cos2','doy_sin3','doy_cos3',
            }
            noise_idx = [j for j,c in enumerate(FEATURES) if c not in NOISE_BLOCK]

            # 에폭별 노이즈 스케줄(처음엔 0.02, 마지막엔 0)
            base_noise = 0.02
            noise_std = base_noise * (1.0 - ep / max(EPOCHS, 1))  # 선형 감쇠

            # ★ anneal 스케줄 (초반 완만, 후반 강하게)
            lam = min(1.0, (ep + 1) / 8.0)       # 8 에폭에 걸쳐 0->1
            alpha_t = 0.3 + 0.4 * lam               # sMAPE 비중: 0.3 -> 0.7

            for i in range(0, len(Xtr), BATCH_SIZE):
                bidx = idx[i:i+BATCH_SIZE]
                Xb, yb = Xtr[bidx], ytr[bidx]
                wdb, mmb, ssb = wdtr[bidx], mmtr[bidx], sstr[bidx]

                # ✅ 연속형 피처에만 노이즈 주입
                if noise_idx and noise_std > 0:
                    Xb[..., noise_idx] = Xb[..., noise_idx] + noise_std * torch.randn_like(Xb[..., noise_idx])

                future_wd = f_wd_tr[bidx]   # (B,H)
                future_mm = f_mm_tr[bidx]   # (B,H)
                future_px = f_px_tr[bidx]   # (B,H)

                model.use_residual_base = True
                # (선택) base_last도 넘겨주고 싶으면:
                base_last = Xb[:, -1, FEAT['rolling_mean_7']]  # (B,)
                pred = model(
                    Xb, wdb, ssb, mmb,
                    future_weekday=future_wd,
                    future_month=future_mm,
                    future_prox=future_px,
                    base_last=base_last
                )
                # #Debug log
                # if ((ep + 1) % 5 == 0) and (i == 0):
                #     y0 = pred[0].detach().float()             # (PREDICT,)
                #     y0_np = y0.cpu().numpy()
                #     # tqdm.write를 쓰면 진행바와 섞이지 않습니다.
                #     tqdm.write(
                #         f"[{key}] ep={ep+1:02d} "
                #         f"yhat[0]={np.round(y0_np, 4).tolist()} "
                #         f"std={float(y0.std().item()):.4f}"
                #     )
                #     # print로 찍고 싶으면 flush도 추가
                #     # print(..., flush=True)

                mom = Xb[:, -1, FEAT['momentum_7']]
                prox= Xb[:, -1, FEAT['holiday_prox']]
                vol = Xb[:, -1, FEAT['vol_14']]
                # 타깃 기반 피크 가중치(현재 배치에서 큰 y 값에 더 큰 가중)
                peak_w = (yb.mean(dim=1) / (yb.mean() + 1e-6)).detach().clamp(0.8, 1.6)

                # 기존 피처기반 w와 결합
                w = (1.0 + 1.2*(F.relu(mom) + 0.6*prox + 0.4*F.relu(vol - 0.3))).clamp(1.0, 2.3)
                w = w * peak_w
                w = w.unsqueeze(1).expand_as(pred)
                zero_w = (yb != 0).float()         # y=0이면 0, 아니면 1
                w = w * (0.2 + 0.8 * zero_w)       # 완전 제거가 불안하면 0.2 정도만 남기기
                # 마지막 시점의 이동평균을 기반으로 eps를 키움 (스케일 공간)
                eps_dyn = (0.25 * Xb[:, -1, FEAT['rolling_mean_7']]).unsqueeze(1).expand_as(pred)
                eps_dyn = eps_dyn.clamp(5e-4, 5e-2)  # 과도 방지

                loss = weighted_huber_smape(pred, yb, w=w,    delta=delta_y, eps=eps_dyn, alpha=alpha_t)


                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if max_grad_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()

                bs = yb.size(0)
                sum += loss.item() * bs
                n_obs += bs

            train_losses.append(sum / max(n_obs,1))

            if use_validation:
                model.eval()
                with torch.no_grad():
                    sum_val, n_val = 0.0, 0
                    for i in range(0, len(Xval), BATCH_SIZE):
                        Xb, yb = Xval[i:i+BATCH_SIZE], yval[i:i+BATCH_SIZE]
                        wdb, ssb = wdval[i:i+BATCH_SIZE], ssval[i:i+BATCH_SIZE]
                        mmb      = mmval[i:i+BATCH_SIZE]
                        future_wd = f_wd_val[i:i+BATCH_SIZE]
                        future_mm = f_mm_val[i:i+BATCH_SIZE]
                        future_px = f_px_val[i:i+BATCH_SIZE]
                        model.use_residual_base = True
                        # (선택) base_last도 넘겨주고 싶으면:
                        base_last = Xb[:, -1, FEAT['rolling_mean_7']]  # (B,)
                        pred = model(Xb, wdb, ssb, mmb, future_wd, future_mm, future_px, base_last=base_last)

                        eps_dyn_val = (0.25 * Xb[:, -1, FEAT['rolling_mean_7']]).unsqueeze(1).expand_as(pred)
                        eps_dyn_val = eps_dyn_val.clamp(5e-4, 5e-2)

                        loss_val = weighted_huber_smape(pred, yb, w=None, delta=delta_y, eps=eps_dyn_val, alpha=alpha_t).item()
                        sum_val += loss_val * yb.size(0)
                        n_val   += yb.size(0)
                        val_sm = smape_loss(pred, yb, eps=1e-3, reduction='mean', ignore_zero_target=True).item()

                    val_loss = sum_val / max(n_val, 1)
                    val_losses.append(val_loss)
                    val_smape_list.append(val_sm)
            
                    # 스케줄러/얼리스탑
                    scheduler.step(val_loss)
                    if early.step(loss_val, model, ep):
                        print(f"[{key}] Early stop @ {ep+1} | best val_unw={min(val_losses):.6f} | best_SMAPE={min(val_smape_list):.4f}")
                        break

        if use_validation:
            visualize_loss(
            train_losses, val_losses, key,
            save=True, show=False, verbose=False,
            val_smape=val_smape_list,
        )
        else:
            visualize_loss(train_losses, None, key, save=True, show=False, verbose=False)

        # ---- 상한선 저장 & 시퀀스 캐시 ----
        lower_bound = float(max(np.quantile(group['매출수량'].values, 0.01), 1.0))
        upper_bound = float(np.quantile(group['매출수량'].values, 0.999))

        trained_models[key] = {
            'model': model.eval(),
            'scaler_y': scaler_y, 'scaler_rm': scaler_rm, 'scaler_delta': scaler_delta,
            'upper_bound': upper_bound,
            'lower_bound' : lower_bound,
            'feature_order': FEATURES,
            'last_sequence': {
                'X_num': ft[FEATURES].values[-LOOKBACK:],
                'weekday': ft['weekday'].values[-LOOKBACK:],
                'season':  ft['season'].values[-LOOKBACK:],
                'month_idx': ft['month_idx'].values[-LOOKBACK:]   # <- 추가
            }
        }

    return trained_models


def visualize_loss(
    train_losses,
    val_losses,
    store_menu,
    save=False,
    out_dir="./loss_plots",
    show=False,
    verbose=False,
    val_smape=None      # sMAPE (scaled)
):
    import numpy as np
    plt.figure(figsize=(6,4))
    ax = plt.gca()  # 메인 축

    # 리스트/텐서 → float 배열 변환
    def to_float_array(xs):
        if xs is None:
            return np.array([], dtype=float)
        try:
            arr = np.asarray([float(x) for x in xs], dtype=float)
        except Exception:
            arr = np.array(xs, dtype=float)
        return arr

    tr = to_float_array(train_losses)
    va = to_float_array(val_losses)

    # 유한값만 마스크
    tr_mask = np.isfinite(tr)
    va_mask = np.isfinite(va)

    drew_any = False

    if tr.size > 0 and tr_mask.any():
        x_tr = np.arange(1, tr.size + 1)[tr_mask]
        y_tr = tr[tr_mask]
        ax.plot(x_tr, y_tr, marker='o', linewidth=1.5, label='Train Loss')
        drew_any = True

    if va.size > 0 and va_mask.any():
        x_va = np.arange(1, va.size + 1)[va_mask]
        y_va = va[va_mask]
        ax.plot(x_va, y_va, marker='o', linewidth=1.5, label='Validation Loss')
        drew_any = True

    title = f"[{store_menu}] Train vs Validation Loss"
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")

    # ===== sMAPE 보조축 =====
    has_smape = val_smape is not None and len(val_smape) > 0
    if has_smape:
        ax2 = ax.twinx()
        if has_smape:
            sm = to_float_array(val_smape)
            msk = np.isfinite(sm)
            if sm.size > 0 and msk.any():
                x_sm = np.arange(1, sm.size + 1)[msk]
                ax2.plot(x_sm, sm[msk], linestyle='--', marker='x', linewidth=1.2,
                         label='Val sMAPE (scaled)')
        ax2.set_ylabel("sMAPE")
        ax2.grid(False)

        # 범례 합치기
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
    else:
        if drew_any:
            ax.legend(loc='upper right')

    if drew_any:
        ax.grid(True, alpha=0.4)
        ymin = min(np.min(tr[tr_mask]) if tr_mask.any() else np.inf,
                   np.min(va[va_mask]) if va_mask.any() else np.inf)
        ymax = max(np.max(tr[tr_mask]) if tr_mask.any() else -np.inf,
                   np.max(va[va_mask]) if va_mask.any() else -np.inf)
        if np.isfinite(ymin) and np.isfinite(ymax) and ymin != ymax:
            pad = 0.05 * (ymax - ymin)
            ax.set_ylim(ymin - pad, ymax + pad)
    else:
        msg = "No points to plot"
        if tr.size == 0 and (va is None or va.size == 0):
            msg += " (empty train/val lists)"
        elif (tr.size > 0 and not tr_mask.any()) and (va.size == 0 or not va_mask.any()):
            msg += " (all values are NaN/Inf)"
        ax.grid(True, alpha=0.4)
        ax.text(0.5, 0.5, msg, ha='center', va='center',
                transform=ax.transAxes, fontsize=12)

    # 파일명 안전 처리
    name_str = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))
    safe_name = re.sub(r'[^\w\-_.]', '_', name_str)

    if verbose:
        print(f"[visualize_loss] {safe_name}: "
              f"len(train)={len(tr)}, finite(train)={tr_mask.sum()}, "
              f"len(val)={len(va)}, finite(val)={va_mask.sum()}, drew={drew_any}")

    if save:
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, f"{safe_name}.png")
        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
    if show and not save:
        plt.tight_layout()
        plt.show()

    plt.close()


def predict_nhits_embed(
    test_df,
    trained_models,
    test_prefix: str,
    *,
    discontinued: dict[str, str | pd.Timestamp] | None = None,  # <- 추가
    rule: str = 'after',     # 'after' : 단종일자 이후(>) 0, 'on_or_after' : 단종일자 당일 포함(>=) 0
    grace_days: int = 0      # 유예일 (단종일자 + grace_days 이후부터 0)
):
    """
    N-HiTS(+weekday/season 임베딩) 예측 파이프라인 (+ 단종 처리)
    - 입력: test_df (영업일자, 영업장명_메뉴명, 매출수량 등 포함)
    - 출력: pd.DataFrame([영업일자, 영업장명_메뉴명, 매출수량])
    - 주의: feature 목록/순서는 train_nhits_embed와 동일해야 함.
    - discontinued: {'영업장명_메뉴명': 'YYYY-MM-DD' 또는 Timestamp} 형태의 단종 딕셔너리
    """
    results = []


    # ----- (A) 단종 딕셔너리 Timestamp 정규화 -----
    cutoff_map = None
    if discontinued is not None:
        def _to_ts(v):
            return v if isinstance(v, pd.Timestamp) else pd.to_datetime(v)
        cutoff_map = {k: _to_ts(v) for k, v in discontinued.items()}
        if grace_days != 0:
            for k in cutoff_map:
                cutoff_map[k] = cutoff_map[k] + pd.Timedelta(days=grace_days)

    for store_menu, store_test in test_df.groupby(['영업장명_메뉴명'], sort=False):
        key = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))
        if key not in trained_models:
            print('key is not equal')
            continue

        pack         = trained_models[key]
        model        = pack['model']
        scaler_y     = pack['scaler_y']
        scaler_rm    = pack['scaler_rm']
        scaler_delta = pack['scaler_delta']
        upper_bound  = pack.get('upper_bound', None)
        lower_bound  = pack.get('lower_bound', 1.0)

        store_test_sorted = store_test.sort_values('영업일자').copy()
        store_test_sorted['영업일자'] = pd.to_datetime(store_test_sorted['영업일자'])
        # 학습과 동일 전처리 (fit=False)
        ft = build_features(store_test_sorted, scaler_y, scaler_rm, scaler_delta, fit=False, date_col='영업일자')

        # 5) 입력 윈도우 구성
        if len(ft) < LOOKBACK:
            last_seq  = pack['last_sequence']
            x_num_np  = np.asarray(last_seq['X_num'], dtype='float32')
            weekday_np= np.asarray(last_seq['weekday'], dtype='int64')
            season_np = np.asarray(last_seq['season'],  dtype='int64')
            month_np  = np.asarray(last_seq['month_idx'], dtype='int64')  # <- 추가
            
        else:
            # (교체) — 숫자 피처와 캘린더/ID를 분리해서 사용
            recent_all  = ft.iloc[-LOOKBACK:].copy()                 # 전체 보존
            recent_feat = recent_all[FEATURES].astype('float32')     # 숫자 피처만
    
            weekday_np = recent_all['weekday'].values[-LOOKBACK:].astype('int64')
            season_np  = recent_all['season'].values[-LOOKBACK:].astype('int64')
            month_np   = recent_all['month_idx'].values[-LOOKBACK:].astype('int64')  # <- 추가
            # 수치 입력은 FEATURES만 사용
            x_num_np   = recent_feat.values
            
        last_obs_date = pd.to_datetime(store_test_sorted['영업일자'].max())
        # 텐서 변환 시 x_month 포함
        x_num_input = torch.tensor(x_num_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        weekday_seq = torch.tensor(weekday_np, dtype=torch.long,   device=DEVICE).unsqueeze(0)
        season_seq  = torch.tensor(season_np,  dtype=torch.long,   device=DEVICE).unsqueeze(0)
        month_seq   = torch.tensor(month_np,   dtype=torch.long,   device=DEVICE).unsqueeze(0)   # <- 추가

          # 6) 예측
        model.eval()
        with torch.no_grad():
            # 1) 예측 구간 실제 날짜
            horizon_dates = pd.date_range(start=last_obs_date + pd.Timedelta(days=1),
                                        periods=PREDICT, freq='D')

            # 2) 요일/월(0~11)
            future_wd = torch.tensor([d.dayofweek for d in horizon_dates],
                                    device=DEVICE, dtype=torch.long).unsqueeze(0)
            future_mm = torch.tensor([d.month - 1 for d in horizon_dates],
                                    device=DEVICE, dtype=torch.long).unsqueeze(0)

            # 3) 휴일 인덱스 & 근접도
            yrs = {d.year for d in horizon_dates}
            yrs.update({min(yrs)-1, max(yrs)+1})
            holiday_index = build_holiday_index(solar_md_holidays, lunar_solar_dates, years=sorted(list(yrs)))

            # 벡터화 근접도
            fut_pd = pd.to_datetime(horizon_dates)
            if len(holiday_index) == 0:
                future_px_np = np.zeros(len(fut_pd), dtype=np.float32)
            else:
                fut_d = fut_pd.values.astype('datetime64[D]')[:, None]
                hol_d = holiday_index.values.astype('datetime64[D]')
                dist = np.abs(fut_d - hol_d).astype('timedelta64[D]').astype(np.int32)
                mind = dist.min(axis=1)
                mind = np.clip(mind, 0, 10)
                future_px_np = ((10 - mind) / 10.0).astype(np.float32)

            future_px = torch.tensor(future_px_np, device=DEVICE).unsqueeze(0)   # (1,H)


            pred_scaled = model(x_num_input, weekday_seq, season_seq, month_seq,
                                future_weekday=future_wd,
                                future_month=future_mm,
                                future_prox=future_px).squeeze(0).cpu().numpy()

        # 7) 역정규화 전 clip 여부
        use_sigmoid = getattr(model, 'use_sigmoid_output', False)
        if use_sigmoid:
            pred_scaled = np.clip(pred_scaled, 0.0, 1.0)

    
        vals_real   = inverse_clipped_from_scaler(scaler_y, pred_scaled)
        #vals_real = np.clip(vals_real, lower_bound, upper_bound) if upper_bound is not None else np.clip(vals_real, lower_bound, None)
        vals_real = np.maximum(vals_real, lower_bound)  
        #vals_real = np.maximum(restored, 1.0)
        # ----- (B) 단종 처리: 예측 구간 실제 날짜와 비교 -----
        # 예측 구간의 "실제 달력 날짜" 생성 (last_obs_date 다음날부터 PREDICT일)
        horizon_dates = pd.date_range(start=last_obs_date + pd.Timedelta(days=1),
                                      periods=PREDICT, freq='D')

        if cutoff_map is not None:
            cutoff = cutoff_map.get(key, None)
            if cutoff is not None:
                if rule == 'on_or_after':
                    zero_mask = horizon_dates >= cutoff
                else:  # 'after'
                    zero_mask = horizon_dates > cutoff
                vals_real = np.where(zero_mask, 0.0, vals_real)

        # 9) 제출 포맷 적재 (네 포맷 그대로)
        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]
        menu_name  = key
        for d, v in zip(pred_dates, vals_real):
            results.append({'영업일자': d, '영업장명_메뉴명': menu_name, '매출수량': float(v)})

    return pd.DataFrame(results, columns=['영업일자','영업장명_메뉴명','매출수량'])




def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    # (영업일자, 메뉴) → 매출수량 딕셔너리로 변환
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']),
        pred_df['매출수량']
    ))

    final_df = sample_submission.copy()

    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:  # 메뉴명들
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)

    return final_df


In [108]:
#Data load
train = pd.read_csv('./train/train.csv')
train = generate_combined_holiday_list(train, solar_md_holidays, lunar_solar_dates)
train = filter_all_menus_by_leading_zeros(
    train,
    min_zero_days=90,
    apply_to_stores=['담하','라그로타','미라시아' ]  # 여기에 대상 업장명만 나열
)
trained_models = train_nhits_embed(train, use_validation=True, plot_dir='./loss_plots_weighted_huber')

Training N-HiTS + Emb:   0%|          | 0/193 [00:00<?, ?it/s]

[느티나무 셀프BBQ_1인 수저세트] Early stop @ 28 | best val_unw=0.384308 | best_SMAPE=0.4808


Training N-HiTS + Emb:   1%|          | 2/193 [00:20<31:17,  9.83s/it]

[느티나무 셀프BBQ_BBQ55(단체)] Early stop @ 28 | best val_unw=0.468763 | best_SMAPE=0.1763


Training N-HiTS + Emb:   2%|▏         | 3/193 [00:29<29:56,  9.46s/it]

...

[담하_한우 미역국 정식] Early stop @ 30 | best val_unw=0.325115 | best_SMAPE=0.6490


Training N-HiTS + Emb:  33%|███▎      | 63/193 [06:21<13:47,  6.36s/it]

[담하_한우 우거지 국밥] Early stop @ 25 | best val_unw=0.274996 | best_SMAPE=0.6509


Training N-HiTS + Emb:  33%|███▎      | 63/193 [06:22<13:08,  6.06s/it]


KeyboardInterrupt: 

In [105]:
all_preds = []

# 모든 test_*.csv 순회
test_files = sorted(glob.glob('./test/TEST_*.csv'))
for path in test_files:
    test_df = pd.read_csv(path)
    # 파일명에서 접두어 추출 (예: TEST_00)
    filename = os.path.basename(path)
    test_prefix = re.search(r'(TEST_\d+)', filename).group(1)

    pred_df = predict_nhits_embed(
        test_df, trained_models, test_prefix,
        discontinued=DISCONTINUED,
        rule='after',      # 단종일 "이후" 0
        grace_days=0       # 유예일 없으면 0
    )
    all_preds.append(pred_df)
    #display(pred_df)
    
full_pred_df = pd.concat(all_preds, ignore_index=True)


In [106]:
sample_submission = pd.read_csv('./sample_submission.csv')
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.to_csv('./Prediction/model_v6_6.csv', index=False, encoding='utf-8-sig')
result = pd.read_csv('./Prediction/model_v6_6.csv')
display(result.head())

/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_14012/1668550076.py:1311: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '9.656142234802246' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_14012/1668550076.py:1311: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '6.229006290435791' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_14012/1668550076.py:1311: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.9525295495986938' has 

,영업일자,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
0,TEST_00+1일,9.656142,1.000000,6.229006,1.95253,1.115705,2.031053,1.000000,6.428438,3.958583,...,7.493320,11.850199,20.819878,9.961416,53.411808,23.698210,4.164320,25.300634,12.077376,12.264859
1,TEST_00+2일,8.307334,47.986301,3.027274,1.00000,1.223968,1.000000,6.711672,6.968676,1.000000,...,6.959752,1.000000,14.840497,8.488737,45.985363,13.320889,10.863372,13.944457,7.406995,3.971035
2,TEST_00+3일,2.080716,50.116161,1.000000,1.00000,1.000000,1.958788,1.000000,3.436869,1.774673,...,2.503253,6.890410,6.789172,8.383701,38.424812,6.105719,5.613109,37.037945,4.827447,2.003730
3,TEST_00+4일,3.626473,57.943493,2.431361,1.00000,1.002379,1.000000,7.022285,4.359089,2.262196,...,1.000000,4.474716,5.770000,4.994186,20.511953,3.810777,8.329196,7.656891,5.841611,3.005076
4,TEST_00+5일,6.460052,102.982407,2.309976,1.00000,1.088282,1.367106,18.062159,6.779971,2.460149,...,1.000000,1.874148,5.630747,3.581506,15.749710,6.359881,1.747570,8.991849,6.358918,1.000000
